<a href="https://colab.research.google.com/github/Swag-Pseudopy/Escaping-Preference-Collapse/blob/main/ad_inv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import os

rng = np.random.default_rng(42)

# ── Helpers ──────────────────────────────────────────────────────────────────

def make_cycle_matrix(k):
    A = np.zeros((k, k))
    for i in range(k):
        A[i, (i+1) % k] =  1.0
        A[(i+1) % k, i] = -1.0
    return A

def nash_md_step(p, A, eta, tau, p_ref):
    logits = eta * (A @ p)
    # KL regularization toward p_ref
    log_p_new = (1 - eta*tau)*np.log(p) + eta*tau*np.log(p_ref) + logits
    log_p_new -= np.max(log_p_new)  # numerical stability
    p_new = np.exp(log_p_new)
    return p_new / p_new.sum()

def excess_energy(p, k):
    return -np.sum(np.log(p)) - k * np.log(k)

def entropy(p):
    return -np.sum(p * np.log(p + 1e-15))

def delta_ref(p_ref, k):
    return -np.sum(np.log(p_ref + 1e-15)) - k * np.log(k)

def theoretical_C_star(eta, tau, E_exp_bar, d_ref):
    return (eta * E_exp_bar + d_ref) / tau

def drifting_pref(t, k, drift_speed, mode="slow"):
    """
    Smoothly drifts p_ref around the simplex.
    mode='slow': drift_speed * t,  mode='fast': 10x faster
    """
    speed = drift_speed if mode == "slow" else drift_speed * 10
    alpha = speed * t
    # Convex combination of two corners, oscillating
    weights = np.ones(k) / k
    weights[0] += 0.4 * np.sin(alpha)
    weights[1] += 0.4 * np.cos(alpha)
    weights = np.abs(weights)
    return weights / weights.sum()

# ── Experiment parameters ─────────────────────────────────────────────────────

k          = 5
A          = make_cycle_matrix(k)
eta        = 0.05
tau        = eta          # stable regime: tau = kappa*eta, kappa=1
n_iter     = 2000
drift_speed = 0.005       # slow: ~one drift cycle per 1257 iterations

# E_exp estimate at Nash eq for theoretical C* (approximated numerically)
p_nash = np.ones(k) / k
E_exp_nash = (k/2) * np.sum(p_nash * (A @ p_nash)**2)  # near 0 at Nash

# ── Run three conditions ──────────────────────────────────────────────────────

conditions = {
    "Static reference\n(baseline)":  {"mode": "static"},
    "Slow drift\n(adiabatic)":        {"mode": "slow"},
    "Fast drift\n(non-adiabatic)":    {"mode": "fast"},
}

results = {}
for label, cfg in conditions.items():
    p = rng.dirichlet(np.ones(k))  # same init for all
    p = p / p.sum()

    C_traj, H_traj, C_theory, pref_delta = [], [], [], []

    for t in range(n_iter):
        if cfg["mode"] == "static":
            p_ref = np.ones(k) / k
        else:
            p_ref = drifting_pref(t, k, drift_speed, mode=cfg["mode"])

        p = nash_md_step(p, A, eta, tau, p_ref)

        d = delta_ref(p_ref, k)
        # Running estimate of E_exp
        E_exp = (k/2) * np.sum(p * (A @ p)**2)
        C_th  = theoretical_C_star(eta, tau, E_exp, d)

        C_traj.append(excess_energy(p, k))
        H_traj.append(entropy(p))
        C_theory.append(C_th)
        pref_delta.append(d)

    results[label] = {
        "C":        np.array(C_traj),
        "H":        np.array(H_traj),
        "C_theory": np.array(C_theory),
        "delta":    np.array(pref_delta),
    }

# ── Plot ──────────────────────────────────────────────────────────────────────

colors = {"Static reference\n(baseline)":  "#2196F3",
          "Slow drift\n(adiabatic)":        "#4CAF50",
          "Fast drift\n(non-adiabatic)":    "#F44336"}

fig = plt.figure(figsize=(13, 8))
gs  = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.35)

ax1 = fig.add_subplot(gs[0, :])   # excess energy, all conditions
ax2 = fig.add_subplot(gs[1, 0])   # entropy
ax3 = fig.add_subplot(gs[1, 1])   # delta_ref (reference drift magnitude)

iters = np.arange(n_iter)

for label, res in results.items():
    c = colors[label]
    ax1.plot(iters, res["C"],        color=c, lw=1.5, label=label)
    ax1.plot(iters, res["C_theory"], color=c, lw=1.0,
             linestyle="--", alpha=0.6)
    ax2.plot(iters, res["H"],        color=c, lw=1.5, label=label)
    ax3.plot(iters, res["delta"],    color=c, lw=1.5, label=label)

ax1.set_title(r"Excess Energy $\tilde{C}_t$ vs. Theoretical $\tilde{C}^*(t)$ "
              r"(dashed)", fontsize=12)
ax1.set_xlabel("Iteration"); ax1.set_ylabel(r"$\tilde{C}_t$")
ax1.legend(fontsize=8, loc="upper right")

ax2.set_title("Shannon Response Entropy $H(p_t)$", fontsize=12)
ax2.set_xlabel("Iteration"); ax2.set_ylabel("$H(p_t)$")
ax2.axhline(np.log(k), color="k", lw=0.8, linestyle=":",
            label="Max entropy")
ax2.legend(fontsize=8)

ax3.set_title(r"Reference Bias $\delta_{\mathrm{ref}}(t)$", fontsize=12)
ax3.set_xlabel("Iteration")
ax3.set_ylabel(r"$\delta_{\mathrm{ref}}$")
ax3.legend(fontsize=8)

fig.suptitle(
    r"Adiabatic Invariance: Stable orbit tracks drifting $p_{\mathrm{ref}}$"
    "\n"
    r"when drift rate $\ll 1/T(C)$  ($k=5$, $\eta=\tau=0.05$)",
    fontsize=13, y=1.01
)

output_dir = "/mnt/user-data/outputs/"
os.makedirs(output_dir, exist_ok=True)

plt.savefig(os.path.join(output_dir, "adiabatic_invariance.pdf"),
            bbox_inches="tight")
plt.savefig(os.path.join(output_dir, "adiabatic_invariance.png"),
            bbox_inches="tight", dpi=150)
plt.close()
print("Saved: adiabatic_invariance.pdf / .png")

Saved: adiabatic_invariance.pdf / .png


In [3]:
"""
Simplex animation: policy orbit under drifting reference policy.
Three panels (static / slow drift / fast drift), saved as GIF.
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as pe

rng = np.random.default_rng(42)

# ── Barycentric helpers ───────────────────────────────────────────────────────

# Vertices of equilateral triangle in 2D
V = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3)/2]])

def to_cart(p):
    """Barycentric (3,) -> Cartesian (2,)"""
    return p @ V

def draw_simplex(ax, title):
    tri = plt.Polygon(V, fill=False, edgecolor="black", lw=2)
    ax.add_patch(tri)
    labels = ["$r_1$", "$r_2$", "$r_3$"]
    offsets = [(-0.07, -0.05), (1.04, -0.05), (0.5, np.sqrt(3)/2 + 0.04)]
    for label, (x, y) in zip(labels, offsets):
        ax.text(x, y, label, fontsize=11, ha="center")
    # Nash equilibrium
    nash = to_cart(np.ones(3)/3)
    ax.plot(*nash, "k+", ms=10, mew=2, zorder=5)
    ax.set_xlim(-0.15, 1.15); ax.set_ylim(-0.12, 1.0)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title, fontsize=10, pad=4)

# ── Dynamics ──────────────────────────────────────────────────────────────────

def make_A():
    return np.array([[0, 1, -1], [-1, 0, 1], [1, -1, 0]], dtype=float)

def nash_md_step(p, A, eta, tau, p_ref):
    log_p = (1 - eta*tau)*np.log(p) + eta*tau*np.log(p_ref) + eta*(A @ p)
    log_p -= np.max(log_p)
    p_new = np.exp(log_p)
    return p_new / p_new.sum()

def drifting_pref(t, speed):
    alpha = speed * t
    w = np.array([1/3 + 0.35*np.sin(alpha),
                  1/3 + 0.35*np.cos(alpha),
                  1/3 - 0.35*np.sin(alpha) - 0.35*np.cos(alpha)])
    w = np.abs(w); return w / w.sum()

# ── Pre-compute trajectories ──────────────────────────────────────────────────

A      = make_A()
eta    = 0.05
tau    = 0.05
n_iter = 600
tail   = 60   # how many past points to show as fading trail

p0 = rng.dirichlet(np.ones(3))

configs = [
    ("Static\n(baseline)",      "static",  "#2196F3"),
    ("Slow drift\n(adiabatic)", "slow",    "#4CAF50"),
    ("Fast drift\n(non-adiabatic)", "fast","#F44336"),
]

trajs, prefs = {}, {}
for label, mode, _ in configs:
    p = p0.copy()
    traj, pref_traj = [], []
    for t in range(n_iter):
        if mode == "static":
            p_ref = np.ones(3)/3
        elif mode == "slow":
            p_ref = drifting_pref(t, 0.006)
        else:
            p_ref = drifting_pref(t, 0.06)
        p = nash_md_step(p, A, eta, tau, p_ref)
        traj.append(to_cart(p))
        pref_traj.append(to_cart(p_ref))
    trajs[label] = np.array(traj)
    prefs[label] = np.array(pref_traj)

# ── Animation ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
fig.suptitle(
    r"Policy orbit on $\Delta^2$ under drifting $p_{\mathrm{ref}}$"
    "\n"
    r"($k{=}3$, $\eta{=}\tau{=}0.05$) — $\mathbf{+}$ marks Nash eq.",
    fontsize=11, y=1.02
)

for ax, (label, _, color) in zip(axes, configs):
    draw_simplex(ax, label)

# Artists per panel
dots, trails, ref_dots, ref_trails = {}, {}, {}, {}
for ax, (label, _, color) in zip(axes, configs):
    trail,    = ax.plot([], [], "-", color=color, lw=1.2, alpha=0.35)
    dot,      = ax.plot([], [], "o", color=color, ms=6, zorder=6)
    ref_trail,= ax.plot([], [], "--", color="gray", lw=0.9, alpha=0.3)
    ref_dot,  = ax.plot([], [], "s", color="gray", ms=5,
                        zorder=5, label="$p_{\\mathrm{ref}}$")
    trails[label]     = trail
    dots[label]       = dot
    ref_trails[label] = ref_trail
    ref_dots[label]   = ref_dot

# legend once
axes[1].legend(handles=[ref_dots[configs[1][0]]],
               loc="upper right", fontsize=8, framealpha=0.7)

time_text = fig.text(0.5, -0.01, "", ha="center", fontsize=9)

def init():
    for label, _, _ in configs:
        for art in [trails[label], dots[label],
                    ref_trails[label], ref_dots[label]]:
            art.set_data([], [])
    time_text.set_text("")
    return []

def update(frame):
    t = frame
    for label, _, color in configs:
        tr  = trajs[label]
        pr  = prefs[label]
        lo  = max(0, t - tail)
        # alpha fading via linewidth trick — use a single segment
        trails[label].set_data(tr[lo:t+1, 0], tr[lo:t+1, 1])
        dots[label].set_data([tr[t, 0]], [tr[t, 1]])
        ref_trails[label].set_data(pr[lo:t+1, 0], pr[lo:t+1, 1])
        ref_dots[label].set_data([pr[t, 0]], [pr[t, 1]])
    time_text.set_text(f"Iteration {t+1}/{n_iter}")
    return (list(trails.values()) + list(dots.values()) +
            list(ref_trails.values()) + list(ref_dots.values()) +
            [time_text])

frames = list(range(0, n_iter, 3))   # skip 3 for speed
ani = animation.FuncAnimation(
    fig, update, frames=frames, init_func=init,
    blit=False, interval=40
)

writer = animation.PillowWriter(fps=25)
out = "/mnt/user-data/outputs/simplex_adiabatic.gif"
ani.save(out, writer=writer, dpi=110)
plt.close()
print(f"Saved: {out}")

Saved: /mnt/user-data/outputs/simplex_adiabatic.gif


In [4]:
"""
Simplex animation: policy orbit under drifting reference policy.
Three panels (static / slow drift / fast drift), saved as GIF.
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as pe

rng = np.random.default_rng(42)

# ── Barycentric helpers ───────────────────────────────────────────────────────

# Vertices of equilateral triangle in 2D
V = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3)/2]])

def to_cart(p):
    """Barycentric (3,) -> Cartesian (2,)"""
    return p @ V

def draw_simplex(ax, title):
    tri = plt.Polygon(V, fill=False, edgecolor="black", lw=2)
    ax.add_patch(tri)
    labels = ["$r_1$", "$r_2$", "$r_3$"]
    offsets = [(-0.07, -0.05), (1.04, -0.05), (0.5, np.sqrt(3)/2 + 0.04)]
    for label, (x, y) in zip(labels, offsets):
        ax.text(x, y, label, fontsize=11, ha="center")
    # Nash equilibrium
    nash = to_cart(np.ones(3)/3)
    ax.plot(*nash, "k+", ms=10, mew=2, zorder=5)
    ax.set_xlim(-0.15, 1.15); ax.set_ylim(-0.12, 1.0)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title, fontsize=10, pad=4)

# ── Dynamics ──────────────────────────────────────────────────────────────────

def make_A():
    return np.array([[0, 1, -1], [-1, 0, 1], [1, -1, 0]], dtype=float)

def nash_md_step(p, A, eta, tau, p_ref):
    log_p = (1 - eta*tau)*np.log(p) + eta*tau*np.log(p_ref) + eta*(A @ p)
    log_p -= np.max(log_p)
    p_new = np.exp(log_p)
    return p_new / p_new.sum()

def drifting_pref(t, speed):
    alpha = speed * t
    w = np.array([1/3 + 0.35*np.sin(alpha),
                  1/3 + 0.35*np.cos(alpha),
                  1/3 - 0.35*np.sin(alpha) - 0.35*np.cos(alpha)])
    w = np.abs(w); return w / w.sum()

# ── Pre-compute trajectories ──────────────────────────────────────────────────

A      = make_A()
eta    = 0.05
tau    = 0.008   # smaller tau -> larger visible orbit
n_iter = 800
tail   = 80

p0 = rng.dirichlet(np.ones(3))

configs = [
    ("Static\n(baseline)",          "static",  "#2196F3"),
    ("Slow drift\n(adiabatic)",     "slow",    "#4CAF50"),
    ("Fast drift\n(non-adiabatic)", "fast",    "#F44336"),
]

trajs, prefs = {}, {}
for label, mode, _ in configs:
    p = p0.copy()
    traj, pref_traj = [], []
    for t in range(n_iter):
        if mode == "static":
            p_ref = np.ones(3)/3
        elif mode == "slow":
            p_ref = drifting_pref(t, 0.008)   # slow
        else:
            p_ref = drifting_pref(t, 0.08)    # fast
        p = nash_md_step(p, A, eta, tau, p_ref)
        traj.append(to_cart(p))
        pref_traj.append(to_cart(p_ref))
    trajs[label] = np.array(traj)
    prefs[label] = np.array(pref_traj)

# ── Animation ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
fig.suptitle(
    r"Policy orbit on $\Delta^2$ under drifting $p_{\mathrm{ref}}$"
    "\n"
    r"($k{=}3$, $\eta{=}\tau{=}0.05$) — $\mathbf{+}$ marks Nash eq.",
    fontsize=11, y=1.02
)

for ax, (label, _, color) in zip(axes, configs):
    draw_simplex(ax, label)

# Artists per panel
dots, trails, ref_dots, ref_trails = {}, {}, {}, {}
for ax, (label, _, color) in zip(axes, configs):
    trail,    = ax.plot([], [], "-", color=color, lw=1.2, alpha=0.35)
    dot,      = ax.plot([], [], "o", color=color, ms=6, zorder=6)
    ref_trail,= ax.plot([], [], "--", color="gray", lw=0.9, alpha=0.3)
    ref_dot,  = ax.plot([], [], "s", color="gray", ms=5,
                        zorder=5, label="$p_{\\mathrm{ref}}$")
    trails[label]     = trail
    dots[label]       = dot
    ref_trails[label] = ref_trail
    ref_dots[label]   = ref_dot

# legend once
axes[1].legend(handles=[ref_dots[configs[1][0]]],
               loc="upper right", fontsize=8, framealpha=0.7)

time_text = fig.text(0.5, -0.01, "", ha="center", fontsize=9)

def init():
    for label, _, _ in configs:
        for art in [trails[label], dots[label],
                    ref_trails[label], ref_dots[label]]:
            art.set_data([], [])
    time_text.set_text("")
    return []

def update(frame):
    t = frame
    for label, _, color in configs:
        tr  = trajs[label]
        pr  = prefs[label]
        lo  = max(0, t - tail)
        # alpha fading via linewidth trick — use a single segment
        trails[label].set_data(tr[lo:t+1, 0], tr[lo:t+1, 1])
        dots[label].set_data([tr[t, 0]], [tr[t, 1]])
        ref_trails[label].set_data(pr[lo:t+1, 0], pr[lo:t+1, 1])
        ref_dots[label].set_data([pr[t, 0]], [pr[t, 1]])
    time_text.set_text(f"Iteration {t+1}/{n_iter}")
    return (list(trails.values()) + list(dots.values()) +
            list(ref_trails.values()) + list(ref_dots.values()) +
            [time_text])

frames = list(range(0, n_iter, 3))   # skip 3 for speed
ani = animation.FuncAnimation(
    fig, update, frames=frames, init_func=init,
    blit=False, interval=40
)

writer = animation.PillowWriter(fps=25)
out = "/mnt/user-data/outputs/simplex_adiabatic.gif"
ani.save(out, writer=writer, dpi=110)
plt.close()
print(f"Saved: {out}")

Saved: /mnt/user-data/outputs/simplex_adiabatic.gif
